In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Dual Binary Extreme Logistic Regressors with Direct Inverse Class Weighting & Sequential Inference (`models/lr_extreme.ipynb`)

This notebook trains **Separate Dual Binary Logistic Regressors** for extreme triage levels with **Direct Inverse Class Frequency Weighting** and **Sequential Inference (ESI 1 first, then ESI 5)**:

### System Architecture & Key Features
1. **Configurable Class Count Regulation**: Allows user-defined per-class keep ratios (`keep_ratios <- c("1" = 1.00, "2" = 1.00, "3" = 1.00, "4" = 1.00, "5" = 1.00)`) to regulate dataset counts per class.
2. **Feature Engineering Inputs**: Uses 13 clinical feature engineered inputs (age, gender, breathing difficulty, dyspnea flags, vital sign anomaly flags).
3. **Direct Inverse Class Frequency Weighting**: Computes exact baseline inverse class frequency weights $w_{\text{inv}} = N_{\text{total}} / N_{\text{class}}$ for ESI 1 and ESI 5 (ESI 1 rows excluded for ESI 5) and applies them directly during training (no multipliers).
4. **Sequential Inference Order**:
   - **1st Inference (Layer 1 - ESI 1 Regressor)**: `if (Model 1 output == "1")` -> Predict **ESI 1**.
   - **2nd Inference (Layer 2 - ESI 5 Regressor)**: `else if (Model 2 output == "5")` -> Predict **ESI 5**.
   - `else` -> Predict **neither** (ESI 2, 3, 4).
5. **Reports & Diagnostic Artifacts**:
   - **CSV Reports**: `reports/lr_extreme_val_report.csv`, `reports/lr_extreme_test_report.csv`.
   - **Model Artifact Exports**: Saved to `deploy/lr_feng_esi1_extreme_model.rds` and `deploy/lr_feng_esi5_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 FE Inputs, Apply Complete Case Analysis & Configurable Class Count Regulation
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Construct 13 Clinical Feature Engineered Inputs
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))

initial_rows <- nrow(df_feng)
df <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df), nrow(df)))

# ---------------------------------------------------------
# CONFIGURABLE DATASET COUNT REGULATION PER CLASS
# Allows user to specify keep ratios or subsampling per ESI level (1 to 5)
# Set keep ratio to 1.0 to keep 100% of rows (no downsampling)
# ---------------------------------------------------------
keep_ratios <- c(
  "1" = 1.00,  # Keep 100% of ESI 1 rows
  "2" = 1.00,  # Keep 100% of ESI 2 rows
  "3" = 1.00,  # Keep 100% of ESI 3 rows
  "4" = 1.00,  # Keep 100% of ESI 4 rows
  "5" = 1.00   # Keep 100% of ESI 5 rows
)

set.seed(config$training$random_state)
kept_indices <- c()
for (cls in names(keep_ratios)) {
  cls_idx <- which(as.character(df$raw_esi) == cls)
  r <- keep_ratios[cls]
  if (length(cls_idx) > 0 && r < 1.0) {
    n_sample <- round(length(cls_idx) * r)
    cls_idx  <- sample(cls_idx, size = n_sample)
  }
  kept_indices <- c(kept_indices, cls_idx)
}

df <- df[sort(kept_indices), ]

# Create Binary Target Columns
raw_esi_comp <- as.character(df$raw_esi)
df$target_esi1 <- factor(ifelse(raw_esi_comp == "1", "1", "not_1"), levels = c("1", "not_1"))
df$target_esi5 <- factor(ifelse(raw_esi_comp == "5", "5", "not_5"), levels = c("5", "not_5"))
df$target_layer1 <- factor(ifelse(raw_esi_comp == "1", "1", ifelse(raw_esi_comp == "5", "5", "neither")),
                           levels = c("1", "5", "neither"))

cat(sprintf("Dataset Ready After Class Count Regulation: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("5-Class ESI Distribution:\n")
print(table(df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df$raw_esi, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$raw_esi, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Calculate Direct Inverse Class Frequency Weights & Train Dual Binary Regressors (No Multipliers)
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Filter out ESI 1 rows for training the ESI 5 model
train_esi5_df <- train_df[train_df$raw_esi != "1", ]

# 1. Calculate Base Inverse Class Frequency Weights
n_total_esi1 <- nrow(train_df)
n_esi1       <- sum(train_df$target_esi1 == "1")
inv_weight_esi1 <- n_total_esi1 / n_esi1

n_total_esi5 <- nrow(train_esi5_df)
n_esi5       <- sum(train_esi5_df$target_esi5 == "5")
inv_weight_esi5 <- n_total_esi5 / n_esi5

cat(sprintf("=== Direct Inverse Class Frequency Weight Calculation ===\n"))
cat(sprintf("  - ESI 1 Regressor (Model 1): %d / %d -> Direct Inverse Weight: %.4f\n", n_esi1, n_total_esi1, inv_weight_esi1))
cat(sprintf("  - ESI 5 Regressor (Model 2, ESI 1 Excluded): %d / %d -> Direct Inverse Weight: %.4f\n\n", n_esi5, n_total_esi5, inv_weight_esi5))
feat_names <- setdiff(names(train_df), c("raw_esi", "target_esi1", "target_esi5", "target_layer1"))
formula_esi1 <- as.formula(paste("target_esi1 ~", paste(feat_names, collapse = " + ")))
formula_esi5 <- as.formula(paste("target_esi5 ~", paste(feat_names, collapse = " + ")))
weights_esi1 <- ifelse(train_df$target_esi1 == "1", inv_weight_esi1, 1.0)
weights_esi5 <- ifelse(train_esi5_df$target_esi5 == "5", inv_weight_esi5, 1.0)
cat("Training ESI 1 Binary Regressor with Direct Inverse Class Weight...\n")
lr_esi1_final <- multinom(formula_esi1, data = train_df, weights = weights_esi1, trace = FALSE, MaxNWts = 5000)
cat("Training ESI 5 Binary Regressor with Direct Inverse Class Weight (ESI 1 Excluded)...\n")
lr_esi5_final <- multinom(formula_esi5, data = train_esi5_df, weights = weights_esi5, trace = FALSE, MaxNWts = 5000)
cat("Dual Binary Regressors Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Sequential Inference Protocol (ESI 1 first -> ESI 5 second) & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_sequential_extreme_cascade <- function(mod1, mod5, data, set_name) {
  N <- nrow(data)
  
  pred1 <- as.character(predict(mod1, newdata = data))
  pred5 <- as.character(predict(mod5, newdata = data))
  
  pred1[is.na(pred1)] <- "not_1"
  pred5[is.na(pred5)] <- "not_5"
  
  cascade_pred <- character(N)
  for (i in 1:N) {
    if (!is.na(pred1[i]) && pred1[i] == "1") {
      cascade_pred[i] <- "1"       # 1st Inference: ESI 1 first
    } else if (!is.na(pred5[i]) && pred5[i] == "5") {
      cascade_pred[i] <- "5"       # 2nd Inference: ESI 5 second
    } else {
      cascade_pred[i] <- "neither" # Fallback to intermediate
    }
  }
  
  target_classes <- c("1", "5", "neither")
  pred_factor   <- factor(cascade_pred, levels = target_classes)
  actual_factor <- factor(data$target_layer1, levels = target_classes)
  
  cm  <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_cls <- cm$byClass[, "Pos Pred Value"]
  rec_by_cls  <- cm$byClass[, "Sensitivity"]
  macro_prec  <- mean(prec_by_cls, na.rm = TRUE)
  macro_rec   <- mean(rec_by_cls,  na.rm = TRUE)
  
  # Probabilities Matrix
  p1_res <- predict(mod1, newdata = data, type = "probs")
  p5_res <- predict(mod5, newdata = data, type = "probs")
  
  p1 <- if (is.matrix(p1_res)) {
    if ("1" %in% colnames(p1_res)) p1_res[, "1"] else 1 - p1_res[, "not_1"]
  } else {
    1 - p1_res
  }
  
  p5 <- if (is.matrix(p5_res)) {
    if ("5" %in% colnames(p5_res)) p5_res[, "5"] else 1 - p5_res[, "not_5"]
  } else {
    1 - p5_res
  }
  
  p_neither <- pmax(0, 1 - (p1 + p5))
  prob_matrix <- cbind("1" = p1, "5" = p5, "neither" = p_neither)
  
  pr_auc_by_cls <- numeric(3)
  names(pr_auc_by_cls) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(actual_factor == cls, 1, 0)
    pr_auc_by_cls[cls] <- calc_pr_auc(act_bin, prob_matrix[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_cls, na.rm = TRUE)
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  actual_table <- table(actual_factor)
  pred_table   <- table(pred_factor)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  class_comparison <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_cls, 4),
    Recall       = round(rec_by_cls, 4),
    PR_AUC       = round(pr_auc_by_cls, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   DUAL LOGISTIC REGRESSORS SEQUENTIAL INFERENCE - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Target Class Count Comparison & Performance Summary Table:\n")
  print(class_comparison)
  
  cat("\nFull Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  return(class_comparison)
}
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
# Evaluate & Export Validation CSV Report
val_report <- evaluate_sequential_extreme_cascade(lr_esi1_final, lr_esi5_final, val_df, "Validation")
write.csv(val_report, file = file.path(reports_dir, "lr_extreme_val_report.csv"), row.names = FALSE)
cat("Validation Evaluation CSV Report written to: reports/lr_extreme_val_report.csv\n")
# Evaluate & Export Test CSV Report
test_report <- evaluate_sequential_extreme_cascade(lr_esi1_final, lr_esi5_final, test_df, "Test")
write.csv(test_report, file = file.path(reports_dir, "lr_extreme_test_report.csv"), row.names = FALSE)
cat("Test Evaluation CSV Report written to: reports/lr_extreme_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Dual Pre-Trained Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_esi1_path <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
model_esi5_path <- file.path(deploy_dir, "lr_feng_esi5_extreme_model.rds")
model_lr_path   <- file.path(deploy_dir, "lr_extreme_model.rds")
saveRDS(list(model = lr_esi1_final, preproc = preproc), file = model_esi1_path)
saveRDS(list(model = lr_esi5_final, preproc = preproc), file = model_esi5_path)
# Save combined list to lr_extreme_model.rds for single-load compatibility
saveRDS(list(model_esi1 = lr_esi1_final, model_esi5 = lr_esi5_final, preproc = preproc), file = model_lr_path)
cat("ESI 1 Extreme Model saved to:", model_esi1_path, "\n")
cat("ESI 5 Extreme Model saved to:", model_esi5_path, "\n")
cat("Combined Dual Extreme Model saved to:", model_lr_path, "\n")